# Colab 18 — Baseline: pretrained-embedding (ESM2) vs our SNN

**Question (from supervision).** A learned edit-distance approximator must beat a *simple* solution: **pretrained embeddings + a logistic/ridge readout on Levenshtein**. Deeper, this is the **data-agnosticism / abstraction-vs-memorisation** probe: does a 35M protein LM (trained on millions of sequences for a *biological* objective) recover edit-distance as well as our 150K-param conv encoder trained *zero-shot* on synthetic edit-distance pairs?

**ESM2 = data-dependent foil, not a competitor.** It encodes *biological* similarity, not string/edit similarity. We measure how far each representation's edit-distance tracking survives **off its home modality** (AA -> SS -> 3Di).

**Design: a clean 2x2 {SNN, ESM2} x {free, +head}.**
- **free** -- zero-shot geometry (SNN L2; ESM2 cosine). Neither sees a natural pair.
- **+head** -- frozen embedding + Ridge/Logistic on `|e_a - e_b|`, **component-grouped 5-fold CV** (union-find on the pair graph; no protein shared across folds). Isolates *embedding quality* from *readout supervision*. **ESM2+head = the target-supervised upper-control baseline** the supervisor asked for; **SNN+head = its matched control**. (Ordinary pair-KFold would leak protein identity and inflate the heads -- we use grouped CV.)

**Metrics.** Pearson r (calibrated tracking) and **AUROC is-high** -- a *pairwise high-sim discrimination proxy*, **not** proof of full-pool retrieval (cf. `cross_rep.md` section 6).

**Scope.** Phase-one AA/SS/3Di. Pairwise (this notebook, Sections 1-7) **and** full-pool set-based retrieval (Sections 8-9, mirroring colab19's exact-Levenshtein oracle) are both covered. The plain-English axis is probed separately in `colab21_natural_language.ipynb`.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR='/content/thesis-edit-distance-nn/sampledata/cath'
CACHE='/content/bench_cache'; os.makedirs(CACHE, exist_ok=True)
for f in ['cath_s20_train70.csv.gz','cath_s20_test30.csv.gz','cath_eval.csv.gz','cath_s20_3di.csv.gz']:
    pth=os.path.join(DATA_DIR,f); print(f'{"OK" if os.path.exists(pth) else "MISSING":<8} {pth}')

In [ ]:
!pip install torch rapidfuzz scikit-learn scipy transformers matplotlib --quiet

In [ ]:
import time, json, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, mean_absolute_error
from rapidfuzz.distance import Levenshtein as RFLev
SEED=42; torch.manual_seed(SEED); np.random.seed(SEED); rng=np.random.default_rng(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:',device)

## 2. Config + helpers (mirrors colab17b)

In [ ]:
AA_ALPHABET='ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET='HLS'
CHAR_TO_IDX={c:i for i,c in enumerate(AA_ALPHABET)}; PAD_IDX=20; VOCAB=21
MIN_LEN=50; MAX_LEN=200; N_TRAIN=30000; EPOCHS=30; BS=128; K=16
BAND_LOW=0.30; BAND_HIGH=0.70
AA_SET=set(AA_ALPHABET); SS_SET=set(SS_ALPHABET)
is_aa=lambda s: all(c in AA_SET for c in s); is_ss=lambda s: all(c in SS_SET for c in s)
def norm_lev(a,b):
    L=max(len(a),len(b)); return 1.0 if L==0 else 1.0-RFLev.distance(a,b)/L
def encode_pad(seq):
    idx=[CHAR_TO_IDX[c] for c in seq][:MAX_LEN]; idx+=[PAD_IDX]*(MAX_LEN-len(idx))
    return torch.tensor(idx,dtype=torch.long)
def perturb(seq,k,abc):
    s=list(seq); abc=list(abc)
    for _ in range(k):
        if len(s)==0: op='ins'
        elif len(s)>=MAX_LEN: op=rng.choice(['sub','del'])
        else: op=rng.choice(['sub','ins','del'])
        if op=='sub': i=rng.integers(0,len(s)); s[i]=rng.choice([c for c in abc if c!=s[i]])
        elif op=='ins': i=rng.integers(0,len(s)+1); s.insert(i,rng.choice(abc))
        else: i=rng.integers(0,len(s)); del s[i]
    return ''.join(s)
def rand_aa(): L=int(rng.integers(MIN_LEN,MAX_LEN+1)); return ''.join(rng.choice(list(AA_ALPHABET),size=L))
def bin_idx(x): return 0 if x<BAND_LOW else (1 if x<BAND_HIGH else 2)

## 3. Pool + eval + component groups (union-find on the eval-pair graph)

In [ ]:
tr=pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'); te=pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')
p=pd.concat([tr,te],ignore_index=True).drop_duplicates('domain_id')
p=p[p.aa_seq.apply(is_aa)]; p=p[p.ss_seq.apply(is_ss)]; p=p[p.aa_seq.str.len()==p.ss_seq.str.len()]
p['L']=p.aa_seq.str.len(); RESCUED={'4z0mC02','3qkaE02'}; inr=p.L.between(MIN_LEN,MAX_LEN)
p=p[inr|p.domain_id.isin(RESCUED)].reset_index(drop=True)
id_to_aa=dict(zip(p.domain_id,p.aa_seq)); id_to_ss=dict(zip(p.domain_id,p.ss_seq))
d3=pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz'); id_to_3di=dict(zip(d3.domain_id,d3['3di'].astype(str)))
ev=pd.read_csv(f'{DATA_DIR}/cath_eval.csv.gz')
ev['3di_score']=[norm_lev(id_to_3di[a],id_to_3di[b]) if (a in id_to_3di and b in id_to_3di) else np.nan
                 for a,b in zip(ev.domain_a,ev.domain_b)]
prots=sorted(set(ev.domain_a)|set(ev.domain_b)); pidx={d:i for i,d in enumerate(prots)}
LOOK={'AA':id_to_aa,'SS':id_to_ss,'3Di':id_to_3di}; SCORE={'AA':'aa_score','SS':'ss_score','3Di':'3di_score'}
# union-find: proteins co-occurring in a pair share a component -> grouped CV cannot leak identity
par=list(range(len(prots)))
def find(x):
    while par[x]!=x: par[x]=par[par[x]]; x=par[x]
    return x
for a,b in zip(ev.domain_a,ev.domain_b):
    ra,rb=find(pidx[a]),find(pidx[b]); par[ra]=rb
pair_group=np.array([find(pidx[a]) for a in ev.domain_a])
print(f'pool={len(p)}  eval_pairs={len(ev)}  proteins={len(prots)}  components={len(set(pair_group))}')

## 4. ESM2 embeddings (frozen, mean-pooled). 35M reproduces the committed numbers; swap a larger variant on GPU.

In [ ]:
from transformers import AutoTokenizer, AutoModel
ESM2_MODEL='facebook/esm2_t12_35M_UR50D'   # e.g. 'facebook/esm2_t33_650M_UR50D' on GPU
tok=AutoTokenizer.from_pretrained(ESM2_MODEL); esm=AutoModel.from_pretrained(ESM2_MODEL).to(device).eval()
esm_params=sum(pp.numel() for pp in esm.parameters())

@torch.no_grad()
def esm_embed(seqs, bs=32):
    order=np.argsort([len(s) for s in seqs]); out=[None]*len(seqs)
    for i in range(0,len(order),bs):
        idx=order[i:i+bs]; batch=[seqs[j] for j in idx]
        enc=tok(batch,return_tensors='pt',padding=True,add_special_tokens=True).to(device)
        h=esm(**enc).last_hidden_state
        mask=enc['attention_mask'].clone(); mask[:,0]=0
        for r,l in enumerate(enc['attention_mask'].sum(1)): mask[r,l-1]=0   # drop cls+eos
        m=mask.unsqueeze(-1).float(); e=(h*m).sum(1)/m.sum(1).clamp(min=1)
        e=F.normalize(e,dim=1).cpu().numpy()
        for kk,j in enumerate(idx): out[j]=e[kk]
    return np.stack(out)

esm_emb={}
for mod in ['AA','SS','3Di']:
    cf=f'{CACHE}/esm2_{mod}.npy'
    esm_emb[mod]=np.load(cf) if os.path.exists(cf) else esm_embed([LOOK[mod][d] for d in prots])
    np.save(cf,esm_emb[mod]); print(mod, esm_emb[mod].shape)

## 5. Our SNN (zero-shot, synthetic-AA only) -- same recipe as colab16b/17b

In [ ]:
class Enc(nn.Module):
    def __init__(s):
        super().__init__(); s.emb=nn.Embedding(VOCAB,32,padding_idx=PAD_IDX)
        s.c1=nn.Conv1d(32,32,3,padding=1); s.c2=nn.Conv1d(32,64,3,padding=1)
        s.pool=nn.AdaptiveAvgPool1d(K); s.fc=nn.Linear(64*K,128)
    def forward(s,x):
        m=(x!=PAD_IDX).float(); e=s.emb(x).permute(0,2,1)
        h=F.relu(s.c1(e)); h=F.relu(s.c2(h)); h=h*m.unsqueeze(1)
        h=s.pool(h).flatten(1); return F.normalize(s.fc(h),p=2,dim=1)
class Clf(nn.Module):
    def __init__(s):
        super().__init__(); s.encoder=Enc()
        s.head=nn.Sequential(nn.Linear(128,64),nn.LeakyReLU(0.01),nn.Linear(64,3))
    def forward(s,a,b): return s.head(torch.abs(s.encoder(a)-s.encoder(b)))
class DS(Dataset):
    def __init__(s,pp): s.p=pp
    def __len__(s): return len(s.p)
    def __getitem__(s,i): a,b,l=s.p[i]; return encode_pad(a),encode_pad(b),bin_idx(l)

In [ ]:
print('generating 30k synthetic AA pairs...')
pairs=[]
while len(pairs)<N_TRAIN:
    sd=rand_aa(); L=len(sd); t=float(rng.uniform(0,1)); k=max(0,int(round((1-t)*L)))
    o=perturb(sd,k,AA_ALPHABET)
    if 1<=len(o)<=MAX_LEN: pairs.append((sd,o,norm_lev(sd,o)))
dl=DataLoader(DS(pairs),batch_size=BS,shuffle=True)
model=Clf().to(device); opt=torch.optim.Adam(model.parameters(),lr=1e-3); crit=nn.CrossEntropyLoss()
snn_params=sum(pp.numel() for pp in model.parameters()); model.train()
for ep in range(1,EPOCHS+1):
    tot=nb=0
    for a,b,y in dl:
        a,b,y=a.to(device),b.to(device),y.to(device)
        loss=crit(model(a,b),y); opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item(); nb+=1
    if ep%5==0 or ep==1: print(f'  epoch {ep}/{EPOCHS} CE {tot/nb:.4f}')
print(f'SNN params={snn_params}  ESM2 params={esm_params}  ratio={esm_params/snn_params:.0f}x')

In [ ]:
@torch.no_grad()
def snn_embed(look):
    model.eval(); out=np.zeros((len(prots),128),dtype=np.float32)
    for i in range(0,len(prots),256):
        ch=prots[i:i+256]; x=torch.stack([encode_pad(look[d]) for d in ch]).to(device)
        out[i:i+len(ch)]=model.encoder(x).cpu().numpy()
    return out
snn_emb={mod:snn_embed(LOOK[mod]) for mod in ['AA','SS','3Di']}

## 6. Metrics: 2x2 {SNN, ESM2} x {free, +head}, component-grouped CV

In [ ]:
def free_metrics(E,a,b,y,hi,is_snn):
    psim=(1.0-np.linalg.norm(E[a]-E[b],axis=1)/2.0) if is_snn else np.sum(E[a]*E[b],axis=1)
    return dict(pearson=float(np.corrcoef(psim,y)[0,1]),spearman=float(spearmanr(psim,y).correlation),
                auroc=float(roc_auc_score(hi,psim)) if 0<hi.sum()<len(hi) else float('nan'))
def head_metrics(E,a,b,y,hi,groups):
    feat=np.abs(E[a]-E[b]); gkf=GroupKFold(5); pred=np.zeros(len(y)); pred_hi=np.zeros(len(y))
    for trn,tst in gkf.split(feat,groups=groups):
        pred[tst]=Ridge(alpha=1.0).fit(feat[trn],y[trn]).predict(feat[tst])
        if 0<hi[trn].sum()<len(trn):
            pred_hi[tst]=LogisticRegression(max_iter=1000,class_weight='balanced').fit(feat[trn],hi[trn]).predict_proba(feat[tst])[:,1]
    return dict(pearson=float(np.corrcoef(pred,y)[0,1]),mae=float(mean_absolute_error(y,pred)),
                auroc=float(roc_auc_score(hi,pred_hi)) if 0<hi.sum()<len(hi) else float('nan'))

res={'snn_params':int(snn_params),'esm_params':int(esm_params),'ratio':float(esm_params/snn_params)}
for mod in ['AA','SS','3Di']:
    sc=SCORE[mod]; m=ev[sc].notna().values; sub=ev[m]
    a=sub.domain_a.map(pidx).values; b=sub.domain_b.map(pidx).values; y=sub[sc].values
    hi=(y>=BAND_HIGH).astype(int); g=pair_group[m]
    res[mod]=dict(n=int(len(y)),n_high=int(hi.sum()),
        snn_free=free_metrics(snn_emb[mod],a,b,y,hi,True), snn_head=head_metrics(snn_emb[mod],a,b,y,hi,g),
        esm_free=free_metrics(esm_emb[mod],a,b,y,hi,False), esm_head=head_metrics(esm_emb[mod],a,b,y,hi,g))
json.dump(res,open(f'{CACHE}/colab18_results.json','w'),indent=2)

print(f'PEARSON r{"":<6}{"SNN free":>10}{"SNN+head":>10}{"ESM2 free":>10}{"ESM2+head":>10}')
for mod in ['AA','SS','3Di']:
    r=res[mod]; print(f'{mod:<8}n_hi={r["n_high"]:<4}{r["snn_free"]["pearson"]:>10.3f}{r["snn_head"]["pearson"]:>10.3f}{r["esm_free"]["pearson"]:>10.3f}{r["esm_head"]["pearson"]:>10.3f}')
print(f'\nAUROC is-high{"":<2}{"SNN free":>10}{"SNN+head":>10}{"ESM2 free":>10}{"ESM2+head":>10}')
for mod in ['AA','SS','3Di']:
    r=res[mod]; print(f'{mod:<14}{r["snn_free"]["auroc"]:>10.3f}{r["snn_head"]["auroc"]:>10.3f}{r["esm_free"]["auroc"]:>10.3f}{r["esm_head"]["auroc"]:>10.3f}')
print(f'\nparams: SNN={snn_params}  ESM2={esm_params}  ratio={esm_params/snn_params:.0f}x')

In [ ]:
import matplotlib.pyplot as plt
mods=['AA','SS','3Di']; x=np.arange(len(mods)); w=0.2
def col(metric,key):
    return [res[m][key][metric] for m in mods]
fig,ax=plt.subplots(1,2,figsize=(13,4))
for ax_i,met,ttl,lo in [(0,'pearson','Pearson r vs true edit-distance',0),(1,'auroc','AUROC is-high (pairwise proxy)',0.8)]:
    ax[ax_i].bar(x-1.5*w,col(met,'snn_free'),w,label='SNN free')
    ax[ax_i].bar(x-0.5*w,col(met,'snn_head'),w,label='SNN+head')
    ax[ax_i].bar(x+0.5*w,col(met,'esm_free'),w,label='ESM2 free')
    ax[ax_i].bar(x+1.5*w,col(met,'esm_head'),w,label='ESM2+head')
    ax[ax_i].set_title(ttl); ax[ax_i].set_xticks(x); ax[ax_i].set_xticklabels(mods); ax[ax_i].set_ylim(lo,1.0); ax[ax_i].legend(fontsize=8)
plt.tight_layout(); plt.savefig('colab18_baseline.png',dpi=120); plt.show()

## 7. Reading (leak-free, grouped CV)

1. **Zero-shot (free vs free): the SNN beats ESM2 on every metric, every modality** -- widest at 3Di Pearson (~0.64 vs 0.31). No CV, no leakage -> the robust headline. The SNN's untrained-on-target geometry tracks edit-distance better than a generic protein LM's. **Data-agnosticism premium, measured.**
2. **Matched readout (both +head, grouped CV) -- the embedding-quality test.** On **discrimination (AUROC)** the SNN embedding wins everywhere (SS ~0.95 vs ~0.88; 3Di ~0.98 vs ~0.95; AA tie). On **calibrated Pearson** the SNN wins SS but **ESM2 edges 3Di** (~0.69 vs ~0.65). At equal supervision: SNN is the better *ranking* substrate; ESM2 is marginally better for *calibrated 3Di regression*.
3. **"Usable but not directly aligned," made precise.** ESM2's 3Di edit-signal is *latent* (free r~0.31 -> head ~0.69, +0.38); the SNN's is *already aligned* (free ~0.64 ~ head ~0.65). Pretrained biology *contains* edit-distance info but needs a readout to surface it; the SNN exposes it directly in distance.
4. **ESM2 is a *strong* baseline, not broken.** No SS collapse (free r~0.75) -- low-entropy edit-distance ~ length+composition, captured even reading "HLS" as nonsense. The free contrast sharpens on higher-entropy 3Di.

**Caveats.** AA Pearson is uninformative (n_high=5, narrow label range; both heads collapse under grouped CV -- read AA on AUROC only, where all ~tie & underpowered). AUROC is a *pairwise discrimination proxy, not full-pool retrieval* -- the full-pool k-NN form is in Sections 8-9 (set-based frac@k vs the exact-Levenshtein oracle, the lens deployment claims must rest on). Scope = phase-one AA/SS/3Di; the plain-English axis is in `colab21_natural_language.ipynb`. ESM2-35M is conservative (bigger lifts AA, stays generic off-modality).

**Net (thesis-safe).** Zero-shot SNN geometry beats raw ESM2 geometry across AA/SS/3Di; at matched supervised readout the SNN embedding still wins discrimination everywhere, ESM2 edges only calibrated 3Di regression -- at ~230x fewer parameters and zero natural-pair supervision. A genuinely useful baseline; not a clean sweep, but the leak-free story is the stronger one.

## 8. Retrieval form — full-pool k-NN vs colab19 set-based oracle

The pairwise AUROC/Pearson above is **not** the deployment claim. Here both **free** embeddings
are used as a k-NN index over the **full pool** (all proteins, not just eval-pair proteins),
scored exactly the way **colab19** does it — against an **exact-Levenshtein oracle**:

- For each query, the true neighbour set = every pool protein with exact `normLev >= 0.70`.
- `median |true_set|` = crowding (how many genuine high-sim neighbours exist).
- `frac@k = retrieved / min(k, |true_set|)` = fraction of the *achievable* top-k true
  neighbours surfaced. **1.0 = fills every achievable slot** (the set ceiling, NOT identical
  ordering to exact Lev). Crowding-aware, comparable across alphabets.

Retrieval uses **free** scores only: **SNN = L2** (`1 - ||e_a-e_b||/2`), **ESM2 = cosine**.
The head is a pairwise classifier, not a retrieval index. This makes the SNN-vs-ESM2 comparison
honest in the crowded SS/3Di alphabets (where designated-single-partner hits@k is ill-posed, cf.
`cross_rep.md` §6).

In [ ]:
from rapidfuzz.process import cdist as rf_cdist

# Full retrieval pool = every protein in `p` (10,117), not just the eval-pair proteins.
all_domains = list(p.domain_id)
print('full retrieval pool:', len(all_domains))

def feed_high_pairs(feed):
    sc, lk = SCORE[feed], LOOK[feed]
    inpool = set(all_domains) & set(lk)
    sub = ev.dropna(subset=[sc])
    return sub[(sub[sc] >= BAND_HIGH) & sub.domain_a.isin(inpool) & sub.domain_b.isin(inpool)]

def build_oracle(feed):
    lk = LOOK[feed]
    pool_ids = [d for d in all_domains if d in lk]
    idx = {d: i for i, d in enumerate(pool_ids)}
    pool_seqs = [lk[d] for d in pool_ids]; lens = np.array([len(s) for s in pool_seqs])
    sub = feed_high_pairs(feed)
    q_ids = sorted({d for pr in sub[['domain_a', 'domain_b']].values for d in pr})
    if not q_ids:
        return dict(feed=feed, pool_ids=pool_ids, idx=idx, q_ids=[], true_sets={})
    D = rf_cdist([lk[q] for q in q_ids], pool_seqs, scorer=RFLev.distance, workers=-1).astype(float)
    true_sets = {}
    for r, q in enumerate(q_ids):
        qi = idx[q]; denom = np.maximum(lens, lens[qi]); denom[denom == 0] = 1
        osim = 1.0 - D[r] / denom; osim[qi] = -np.inf
        true_sets[q] = set(np.where(osim >= BAND_HIGH)[0].tolist())
    nt = [len(true_sets[q]) for q in q_ids]
    print(f'  oracle[{feed}]: {len(q_ids)} queries, median |true_set|={int(np.median(nt))}, max={max(nt)}')
    return dict(feed=feed, pool_ids=pool_ids, idx=idx, q_ids=q_ids, true_sets=true_sets)

print('Building exact-Levenshtein oracle neighbour sets over the full pool...')
ORACLE = {f: build_oracle(f) for f in ['AA', 'SS', '3Di']}

In [ ]:
@torch.no_grad()
def snn_pool_emb(feed):
    look = LOOK[feed]; ids = ORACLE[feed]['pool_ids']
    out = np.zeros((len(ids), 128), dtype=np.float32); model.eval()
    for i in range(0, len(ids), 256):
        ch = ids[i:i+256]; x = torch.stack([encode_pad(look[d]) for d in ch]).to(device)
        out[i:i+len(ch)] = model.encoder(x).cpu().numpy()
    return out

def esm_pool_emb(feed):
    """Cache keyed on (model name, exact ordered pool ids). A length-only check can silently
    return embeddings aligned to a different id order / model — so we persist a sidecar fingerprint
    and recompute on any mismatch."""
    look = LOOK[feed]; ids = ORACLE[feed]['pool_ids']
    cf = f'{CACHE}/esm2_pool_{feed}.npy'; sf = f'{CACHE}/esm2_pool_{feed}.meta.json'
    if os.path.exists(cf) and os.path.exists(sf):
        meta = json.load(open(sf)); e = np.load(cf)
        if meta.get('model') == ESM2_MODEL and meta.get('ids') == ids and len(e) == len(ids):
            return e
        print(f'  [{feed}] cache fingerprint mismatch (model/ids changed) -> recomputing')
    e = esm_embed([look[d] for d in ids])
    np.save(cf, e); json.dump({'model': ESM2_MODEL, 'ids': ids}, open(sf, 'w'))
    return e

# Both embedding tables are L2-normalised -> SNN ranks by L2, ESM2 by cosine (= dot).
SNN_POOL = {f: snn_pool_emb(f) for f in ['AA', 'SS', '3Di']}
print('SNN pool embeddings done.')
ESM_POOL = {f: esm_pool_emb(f) for f in ['AA', 'SS', '3Di']}
print('ESM2 pool embeddings done.')

In [ ]:
K_LIST = (1, 10, 50)

def retrieval_frac(emb, feed, is_snn, k_list=K_LIST):
    cache = ORACLE[feed]
    if not cache['q_ids']:
        return {'median_n_true': np.nan, 'n_q': 0, **{k: np.nan for k in k_list}}
    idx = cache['idx']; acc = {k: [] for k in k_list}; nts = []
    for q in cache['q_ids']:
        qi = idx[q]; ts = cache['true_sets'][q]; nt = len(ts); nts.append(nt)
        if nt == 0: continue
        if is_snn:
            sim = 1.0 - np.linalg.norm(emb - emb[qi], axis=1) / 2.0
        else:
            sim = emb @ emb[qi]                       # cosine (rows are unit-norm)
        sim[qi] = -np.inf; order = np.argsort(-sim)
        for k in k_list:
            ret = len(set(order[:k].tolist()) & ts); acc[k].append(ret / min(k, nt))
    return {'median_n_true': float(np.median(nts)), 'n_q': len(nts),
            **{k: (float(np.mean(acc[k])) if acc[k] else np.nan) for k in k_list}}

ret_res = {}
print('=' * 78)
print('RETRIEVAL FORM — set-based frac@k vs exact-Levenshtein oracle (full pool)')
print('=' * 78)
print(f'{"model x feed":<20}{"n_q":>5}{"med|T|":>8}{"frac@1":>9}{"frac@10":>9}{"frac@50":>9}')
print('-' * 78)
for feed in ['AA', 'SS', '3Di']:
    rs = retrieval_frac(SNN_POOL[feed], feed, True)
    re = retrieval_frac(ESM_POOL[feed], feed, False)
    ret_res[feed] = {'SNN': rs, 'ESM2': re}
    for name, r in [('SNN-free', rs), ('ESM2-free', re)]:
        print(f"{name+' x '+feed:<20}{r['n_q']:>5}{('%d'%r['median_n_true']) if not np.isnan(r['median_n_true']) else 'n/a':>8}"
              f"{r[1]:>9.3f}{r[10]:>9.3f}{r[50]:>9.3f}")

json.dump(ret_res, open(f'{CACHE}/colab18_retrieval.json', 'w'), indent=2, default=float)
print('\nsaved', f'{CACHE}/colab18_retrieval.json')
print('frac@k = retrieved / min(k, |true_set|); 1.0 = every achievable slot filled with a')
print('genuine high-sim neighbour. med|T| huge (SS/3Di) = crowded alphabet, not model failure.')

## 9. Reading the retrieval form

Same lens as colab19's set-based oracle, now SNN-free vs ESM2-free over the full pool.

- **AA (well-posed, `med|T|` ~ 1):** both should fill the achievable slots — the partner is
  unique. This is the deployment task the thesis actually claims.
- **SS / 3Di (crowded, large `med|T|`):** `frac@k` reads the honest question — *of the genuine
  high-sim neighbours, how many do we surface* — instead of the ill-posed designated-partner
  hits@k. A low number here driven by huge `med|T|` is **task crowding** (alphabet entropy),
  not an embedding failure (cf. `cross_rep.md` §6).
- **SNN vs ESM2:** read whether the SNN's pairwise-AUROC edge (Section 6) does or does not carry
  to full-pool set retrieval. The honest framing for the thesis: efficiency (227× fewer params)
  + zero-shot + pure edit-distance proxy — **not** a retrieval sweep.

This section computes everything in-notebook (no external numbers). The ESM2 pool embeddings are
cached to `bench_cache/` so re-runs are fast.

## 10. Encoding throughput — SNN vs ESM2 (the efficiency headline)

Where the 227× parameter gap actually shows up in wall-clock. Retrieval itself is *identical*
vector search for both models, so the cost difference lives entirely in the **encode step**
(string -> vector, done once per protein, then amortised over every query).

Controlled apples-to-apples: the **same fixed sample** of pool sequences encoded by each model on
the same `device`, with warmup + repeated timings (cache bypassed). Reports total, per-sequence,
throughput, and the ESM2/SNN time ratio next to the parameter ratio. Sample size + repeats auto-
shrink on CPU so this never hangs. (GPU is the deployment-relevant number; CPU is the worst case
for the SNN's advantage since ESM2 has no SIMD/batched edge there.)

In [ ]:
import time

# Auto-scale so CPU runs don't hang (ESM2 forward is heavy without a GPU).
if device.type == 'cuda':
    N_TIME, WARMUP, REPEATS = min(2000, len(all_domains)), 1, 3
else:
    N_TIME, WARMUP, REPEATS = min(400, len(all_domains)), 1, 1
print(f'timing on {device.type}: N={N_TIME} sequences, warmup={WARMUP}, repeats={REPEATS}')

sample_seqs = [id_to_aa[d] for d in all_domains[:N_TIME]]

@torch.no_grad()
def snn_encode(seqs, batch=256):
    model.eval(); out = []
    for i in range(0, len(seqs), batch):
        x = torch.stack([encode_pad(s) for s in seqs[i:i+batch]]).to(device)
        out.append(model.encoder(x).cpu().numpy())
    return np.concatenate(out)

def time_encode(fn, seqs, warmup=WARMUP, repeats=REPEATS):
    for _ in range(warmup): fn(seqs)
    if device.type == 'cuda': torch.cuda.synchronize()
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter(); fn(seqs)
        if device.type == 'cuda': torch.cuda.synchronize()
        ts.append(time.perf_counter() - t0)
    return float(np.mean(ts)), float(np.std(ts))

snn_t, snn_sd = time_encode(snn_encode, sample_seqs)
esm_t, esm_sd = time_encode(lambda s: esm_embed(s), sample_seqs)

def line(name, t, sd):
    return (f'  {name:<5}: {t*1e3:8.1f} +/- {sd*1e3:5.1f} ms total | '
            f'{t/N_TIME*1e3:7.3f} ms/seq | {N_TIME/t:8.0f} seq/s')
print(f'\nEncoded {N_TIME} sequences on {device.type.upper()}:')
print(line('SNN', snn_t, snn_sd)); print(line('ESM2', esm_t, esm_sd))
print(f'\n  encode-time ratio (ESM2 / SNN): {esm_t/snn_t:6.1f}x')
print(f'  parameter ratio  (ESM2 / SNN): {esm_params/snn_params:6.0f}x')
print(f'\n  full-pool ({len(all_domains)}) encode est: '
      f'SNN ~{snn_t/N_TIME*len(all_domains):.1f}s | ESM2 ~{esm_t/N_TIME*len(all_domains):.1f}s')

timing = {'device': device.type, 'n_seqs': N_TIME, 'repeats': REPEATS,
          'snn_total_s': snn_t, 'snn_per_seq_ms': snn_t/N_TIME*1e3, 'snn_seq_per_s': N_TIME/snn_t,
          'esm_total_s': esm_t, 'esm_per_seq_ms': esm_t/N_TIME*1e3, 'esm_seq_per_s': N_TIME/esm_t,
          'encode_time_ratio': esm_t/snn_t, 'param_ratio': esm_params/snn_params,
          'snn_params': int(snn_params), 'esm_params': int(esm_params)}
json.dump(timing, open(f'{CACHE}/colab18_timing.json', 'w'), indent=2, default=float)
print('\nsaved', f'{CACHE}/colab18_timing.json')

**Reading.** `encode-time ratio` is the deployment-relevant efficiency number; compare it to the
`parameter ratio` (227×). They need not match — throughput depends on batching, sequence length,
and kernel efficiency, not just parameter count. The full-pool estimate linearly extrapolates the
per-sequence cost. Pair this with the per-query retrieval cost from `BENCHMARKS.md` Table 3 (Lev DP
vs SNN forward) for the end-to-end *encode-once + search* story. **Caveat:** GPU is the deployment
number; ESM2's relative cost is *lower* on GPU than CPU, so report the device alongside the ratio.